# Integrate samples

After performing quality control, ambient RNA decontamination, and doublet detection for each sample,  
we now integrate the filtered datasets into a unified object for downstream analysis.


In [ ]:
# Loading packages from omicverse and scanpy (Python)
import scanpy as sc
import os
from datetime import date

In [ ]:

file_path = r"F:\Data\snRNAseq\PlaceCell\FilteredData"

samples = {
    sid: os.path.join(file_path, f"{sid}_count_martix_QC_20260119.h5ad")
    for sid in ["PC92", "PC94", "PC103", "PC105", "PC106", "PC110", "PC112", "PC114", "PC118", "PC132"]
}

adatas = {}

for sid, path in samples.items():
    print(f"🔹 Reading sample {sid} from: {path}")
    ad = sc.read_h5ad(path)
    ad.obs["sample"] = sid
    ad.var_names_make_unique()
    adatas[sid] = ad

In [ ]:
adata = sc.concat(adatas, label="sample", join="outer")
adata

In [ ]:
import numpy as np
adata.X[np.isnan(adata.X)] = 0

In [ ]:
import numpy as np
nan_count = np.isnan(adata.X).sum()
print(f"Number of NaN values in adata.X: {nan_count}")
print(f"Max value in martix:" ,adata.X.max())

In [ ]:
adata.obs[["sample","property","condition"]].drop_duplicates()

In [ ]:
adata.obs_names_make_unique()
print(adata.obs["sample"].value_counts())
adata

In [ ]:
adata.layers["counts"] = adata.X.copy()  

In [ ]:
today = date.today().strftime("%Y%m%d")
output_path = os.path.join(file_path, f"PCAnalysis_All_QC_{today}.h5ad")

adata.write_h5ad(output_path, compression="gzip")
print(f"Successful saving：{output_path}")